In [2]:
# ============================================================
# INDIAN QUANT PORTFOLIO & RISK ENGINE
# STEP 3: PORTFOLIO OPTIMIZATION ENGINE
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy.optimize import minimize


# ============================================================
# CONFIGURATION
# ============================================================

DATA_DIR = "data/raw"
RISK_DIR = "data/processed"
OUTPUT_DIR = "data/processed/optimization"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRADING_DAYS = 252

# Portfolio constraints
MIN_WEIGHT = 0.00
MAX_WEIGHT = 0.15

# Initial risk-free rate.
# We will later replace this with a proper Indian
# risk-free rate series if required.
RISK_FREE_RATE = 0.00


# ============================================================
# LOAD DATA
# ============================================================

returns = pd.read_csv(
    os.path.join(DATA_DIR, "daily_returns.csv"),
    index_col="Date",
    parse_dates=True
)

returns = returns.sort_index()

# Remove completely empty observations
returns = returns.dropna(how="all")


# ============================================================
# SELECT INVESTMENT UNIVERSE
# ============================================================

# Only use stocks with sufficient history.
MIN_OBSERVATIONS = int(len(returns) * 0.90)

valid_stocks = [
    col
    for col in returns.columns
    if returns[col].count() >= MIN_OBSERVATIONS
]

returns = returns[valid_stocks]

# Drop remaining missing observations for optimization.
# For covariance estimation we need a common sample.
clean_returns = returns.dropna()

print("=" * 70)
print("PORTFOLIO OPTIMIZATION ENGINE")
print("=" * 70)

print(f"\nStocks used: {len(valid_stocks)}")
print(f"Observations: {len(clean_returns)}")

print("\nUniverse:")
print(valid_stocks)


# ============================================================
# EXPECTED RETURNS
# ============================================================

# Historical annualized mean return

expected_returns = (
    clean_returns.mean() * TRADING_DAYS
)


# ============================================================
# COVARIANCE MATRIX
# ============================================================

covariance_matrix = (
    clean_returns.cov() * TRADING_DAYS
)


# ============================================================
# EWMA COVARIANCE
# ============================================================

ewma_cov_path = os.path.join(
    RISK_DIR,
    "ewma_covariance_matrix.csv"
)

if os.path.exists(ewma_cov_path):

    ewma_covariance = pd.read_csv(
        ewma_cov_path,
        index_col=0
    )

    # Keep only stocks in current universe
    ewma_covariance = ewma_covariance.loc[
        valid_stocks,
        valid_stocks
    ]

else:

    print(
        "\nEWMA covariance not found. "
        "Using standard covariance."
    )

    ewma_covariance = covariance_matrix.copy()


# ============================================================
# PORTFOLIO FUNCTIONS
# ============================================================

def portfolio_return(weights, expected_returns):

    return np.dot(
        weights,
        expected_returns
    )


def portfolio_volatility(weights, covariance):

    return np.sqrt(
        weights.T
        @ covariance
        @ weights
    )


def portfolio_variance(weights, covariance):

    return (
        weights.T
        @ covariance
        @ weights
    )


def portfolio_sharpe(
    weights,
    expected_returns,
    covariance,
    risk_free_rate=0.0
):

    ret = portfolio_return(
        weights,
        expected_returns
    )

    vol = portfolio_volatility(
        weights,
        covariance
    )

    if vol == 0:
        return 0

    return (
        ret - risk_free_rate
    ) / vol


# ============================================================
# CONSTRAINTS
# ============================================================

n_assets = len(valid_stocks)

bounds = [
    (MIN_WEIGHT, MAX_WEIGHT)
    for _ in range(n_assets)
]

constraint_sum = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

constraints = [
    constraint_sum
]


# ============================================================
# INITIAL WEIGHTS
# ============================================================

equal_weights = np.ones(
    n_assets
) / n_assets


# ============================================================
# 1. EQUAL WEIGHT PORTFOLIO
# ============================================================

equal_weight = equal_weights.copy()


# ============================================================
# 2. MINIMUM VARIANCE
# ============================================================

def min_variance_objective(
    weights,
    covariance
):

    return portfolio_variance(
        weights,
        covariance
    )


min_variance_result = minimize(
    min_variance_objective,
    equal_weights,
    args=(covariance_matrix,),
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={
        "maxiter": 2000,
        "ftol": 1e-10
    }
)

if not min_variance_result.success:

    print(
        "\nWARNING: Minimum variance "
        "optimization did not converge."
    )

minimum_variance_weights = (
    min_variance_result.x
)


# ============================================================
# 3. MAXIMUM SHARPE
# ============================================================

def negative_sharpe(
    weights,
    expected_returns,
    covariance
):

    return -portfolio_sharpe(
        weights,
        expected_returns,
        covariance,
        RISK_FREE_RATE
    )


max_sharpe_result = minimize(
    negative_sharpe,
    equal_weights,
    args=(
        expected_returns.values,
        covariance_matrix.values
    ),
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={
        "maxiter": 5000,
        "ftol": 1e-10
    }
)

if not max_sharpe_result.success:

    print(
        "\nWARNING: Maximum Sharpe "
        "optimization did not converge."
    )

maximum_sharpe_weights = (
    max_sharpe_result.x
)


# ============================================================
# 4. RISK PARITY
# ============================================================

def risk_contribution(
    weights,
    covariance
):

    portfolio_vol = portfolio_volatility(
        weights,
        covariance
    )

    marginal_contribution = (
        covariance @ weights
    )

    contribution = (
        weights * marginal_contribution
    )

    return contribution / portfolio_vol


def risk_parity_objective(
    weights,
    covariance
):

    contributions = risk_contribution(
        weights,
        covariance
    )

    target = 1 / len(weights)

    return np.sum(
        (contributions - target) ** 2
    )


risk_parity_result = minimize(
    risk_parity_objective,
    equal_weights,
    args=(covariance_matrix.values,),
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={
        "maxiter": 5000,
        "ftol": 1e-12
    }
)

if not risk_parity_result.success:

    print(
        "\nWARNING: Risk parity "
        "optimization did not converge."
    )

risk_parity_weights = (
    risk_parity_result.x
)


# ============================================================
# 5. MAXIMUM DIVERSIFICATION
# ============================================================

individual_volatility = np.sqrt(
    np.diag(covariance_matrix.values)
)


def diversification_ratio(
    weights,
    covariance,
    asset_volatility
):

    portfolio_vol = portfolio_volatility(
        weights,
        covariance
    )

    weighted_asset_volatility = (
        weights @ asset_volatility
    )

    if portfolio_vol == 0:
        return 0

    return (
        weighted_asset_volatility
        / portfolio_vol
    )


def negative_diversification(
    weights,
    covariance,
    asset_volatility
):

    return -diversification_ratio(
        weights,
        covariance,
        asset_volatility
    )


max_diversification_result = minimize(
    negative_diversification,
    equal_weights,
    args=(
        covariance_matrix.values,
        individual_volatility
    ),
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={
        "maxiter": 5000,
        "ftol": 1e-12
    }
)

if not max_diversification_result.success:

    print(
        "\nWARNING: Maximum diversification "
        "optimization did not converge."
    )

maximum_diversification_weights = (
    max_diversification_result.x
)


# ============================================================
# CREATE WEIGHT DATAFRAME
# ============================================================

weights = pd.DataFrame({

    "Equal_Weight": equal_weight,

    "Minimum_Variance":
        minimum_variance_weights,

    "Maximum_Sharpe":
        maximum_sharpe_weights,

    "Risk_Parity":
        risk_parity_weights,

    "Maximum_Diversification":
        maximum_diversification_weights

}, index=valid_stocks)


# ============================================================
# CLEAN NUMERICAL NOISE
# ============================================================

weights[
    weights.abs() < 1e-8
] = 0


# Re-normalize
for column in weights.columns:

    weights[column] = (
        weights[column]
        / weights[column].sum()
    )


# ============================================================
# PORTFOLIO METRICS
# ============================================================

def calculate_portfolio_metrics(
    weights_vector,
    expected_returns,
    covariance
):

    ret = portfolio_return(
        weights_vector,
        expected_returns
    )

    vol = portfolio_volatility(
        weights_vector,
        covariance
    )

    sharpe = (
        (ret - RISK_FREE_RATE) / vol
        if vol > 0
        else np.nan
    )

    risk_contrib = risk_contribution(
        weights_vector,
        covariance
    )

    diversification = diversification_ratio(
        weights_vector,
        covariance,
        np.sqrt(np.diag(covariance))
    )

    return {
        "Expected_Return": ret,
        "Volatility": vol,
        "Sharpe_Ratio": sharpe,
        "Maximum_Weight":
            np.max(weights_vector),
        "Number_of_Positions":
            np.sum(weights_vector > 0.001),
        "Diversification_Ratio":
            diversification,
        "Max_Risk_Contribution":
            np.max(risk_contrib),
        "Min_Risk_Contribution":
            np.min(risk_contrib)
    }


# ============================================================
# EVALUATE ALL PORTFOLIOS
# ============================================================

portfolio_metrics = {}

for portfolio_name in weights.columns:

    portfolio_metrics[portfolio_name] = (
        calculate_portfolio_metrics(
            weights[portfolio_name].values,
            expected_returns.values,
            covariance_matrix.values
        )
    )


portfolio_metrics = pd.DataFrame(
    portfolio_metrics
).T


# ============================================================
# RISK CONTRIBUTIONS
# ============================================================

risk_contributions = pd.DataFrame(
    index=valid_stocks
)

for portfolio_name in weights.columns:

    rc = risk_contribution(
        weights[portfolio_name].values,
        covariance_matrix.values
    )

    risk_contributions[
        portfolio_name
    ] = rc


# ============================================================
# SAVE OUTPUTS
# ============================================================

weights_path = os.path.join(
    OUTPUT_DIR,
    "portfolio_weights.csv"
)

metrics_path = os.path.join(
    OUTPUT_DIR,
    "portfolio_metrics.csv"
)

risk_contribution_path = os.path.join(
    OUTPUT_DIR,
    "risk_contributions.csv"
)

weights.to_csv(weights_path)

portfolio_metrics.to_csv(
    metrics_path
)

risk_contributions.to_csv(
    risk_contribution_path
)


# ============================================================
# SAVE COVARIANCE USED BY OPTIMIZER
# ============================================================

covariance_matrix.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "optimizer_covariance_matrix.csv"
    )
)

expected_returns.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "optimizer_expected_returns.csv"
    )
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 70)
print("PORTFOLIO METRICS")
print("=" * 70)

print(
    portfolio_metrics.round(4)
)


print("\n" + "=" * 70)
print("PORTFOLIO WEIGHTS")
print("=" * 70)

print(
    weights
    .round(4)
    .sort_values(
        "Maximum_Sharpe",
        ascending=False
    )
)


print("\n" + "=" * 70)
print("RISK CONTRIBUTIONS")
print("=" * 70)

print(
    risk_contributions
    .round(4)
)


print("\n" + "=" * 70)
print("WEIGHT CHECK")
print("=" * 70)

print(
    weights.sum()
)

print("\nMaximum individual weights:")

print(
    weights.max()
)


print("\n" + "=" * 70)
print("OPTIMIZATION COMPLETE")
print("=" * 70)

print(f"\nOutputs saved to:")
print(OUTPUT_DIR)

PORTFOLIO OPTIMIZATION ENGINE

Stocks used: 15
Observations: 2874

Universe:
['KOTAKBANK.NS', 'INFY.NS', 'WIPRO.NS', 'TECHM.NS', 'ONGC.NS', 'NTPC.NS', 'ITC.NS', 'NESTLEIND.NS', 'BRITANNIA.NS', 'SUNPHARMA.NS', 'DRREDDY.NS', 'APOLLOHOSP.NS', 'BHARTIARTL.NS', 'TATASTEEL.NS', 'DLF.NS']

PORTFOLIO METRICS
                         Expected_Return  Volatility  Sharpe_Ratio  \
Equal_Weight                      0.1565      0.1543        1.0138   
Minimum_Variance                  0.1411      0.1372        1.0284   
Maximum_Sharpe                    0.1846      0.1523        1.2122   
Risk_Parity                       0.1680      0.1939        0.8665   
Maximum_Diversification           0.1463      0.1406        1.0400   

                         Maximum_Weight  Number_of_Positions  \
Equal_Weight                     0.0667                 15.0   
Minimum_Variance                 0.1500                 13.0   
Maximum_Sharpe                   0.1500                 11.0   
Risk_Parity          